In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request


import warnings
warnings.filterwarnings("ignore")


# In[2]:


# # creating engines and connections





db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')









# In[4]:


gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)


# In[5]:


def haversine(L1,L2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """

    lon1=L1[0]
    lat1=L1[1]
    lon2=L2[0]
    lat2=L2[1]
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
    return c * r


# In[6]:


# In[6]:

In [ ]:

def loc_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]

    try:
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(vid)])
        #print("LOC PINGS A GAYI")
        
    except Exception as e:
        print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df


In [ ]:
def hb_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    
    try:        
        url=GPS_API_BASE+"/heartbeat/list?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        hb_data=json.load(urllib.request.urlopen(url))
        hb=pd.DataFrame(hb_data['data'][str(vid)])
        #print("HB AAYA")

    except Exception as e:

        print("hb pings issue >> " + str(e))
        hb=pd.DataFrame()
        
    return hb



In [ ]:
# In[8]:


def analysis_flow(l):
    lc=loc_pings(l)
    hb=hb_pings(l)

    lc1=lc.copy()
    hb1=hb.copy()
    
    lc2=lc.copy()
    hb2=hb.copy()

    vid=l[0]
    if (len(hb)>5) & (len(lc)>5):
        device_health=hb_lc(hb1,lc1,l)
        wire_issue=wire_issues(l,hb2,lc2)
        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S')+"- vehicle_id:"+str(vid)+" in analysis flow, went into hb_lc")
        return (device_health,wire_issue)
        
    elif (len(lc)>5) & (len(hb)<5):
        device_health=lc_only(lc1,l)
        wire_issue=wire_issues(l,hb2,lc2)
        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S')+"- vehicle_id:"+str(vid)+" in analysis flow, went into lc_only")
        return (device_health,wire_issue)
        
    elif (len(lc)<5) & (len(hb)>5):
        device_health=hb_only(hb1,l)
        wire_issue=wire_issues(l,hb2,lc2)
        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S')+"- vehicle_id:"+str(vid)+" in analysis flow, went into hb_only")
        return (device_health,wire_issue)
    
    elif (len(lc)<5) & (len(hb)<5):
        device_health=no_data(l)
        wire_issue=wire_issues(l,hb1,lc1)
        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S')+"- vehicle_id:"+str(vid)+" , no_data")
        return (device_health,wire_issue)


# In[9]:


def wire_issues(l, hbc, lcc):
    hb=hbc.copy()
    lc=lcc.copy()
    if (len(hb)>5) & (len(lc)>5):
        day_wire_issue=hb_lc_wire_issue(hb,lc)
        return day_wire_issue
    elif (len(lc)>5) & (len(hb)<1):
        day_wire_issue=lc_only_wire_issue(lc)
        return day_wire_issue
        
    else:
        print("not_eough_data in wire issue" + str(l))
        return None


# In[10]:


def hb_lc_wire_issue(hbc,dfc):
    hb=hbc.copy()
    df=dfc.copy()
    #if 1>0:
    try:

        hb.sort_values('created',inplace=True)
    
        hb.created=hb.created/1000
        hb.created=hb.created.apply(int)
        del df['created']
        del df['vehicleId']
        del hb['updated']
        df.sort_values('time',inplace=True)
    
        if len(hb)<len(df):
        
            df=pd.merge_asof(left=df,right=hb,left_on='time',right_on='created',direction='nearest')
        else:
            df=pd.merge_asof(left=hb,right=df,left_on='created',right_on='time',direction='nearest')
        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(-1)
        df=df[1:]
        time_diff_cons=df['time1']-df['time']
        df.loc[:,'time_diff_cons_pings']=time_diff_cons
    
    
        loc_list=df[['location','loc2']].values.tolist()
        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)

        df.loc[:,'distance']=dis
    
    
        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)
    
        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()
        day=getDay((df['time'].min()+df['time'].max())/2)
    
        #### WIRE CUT KE PARAMETERS ####
        charge_off_Speed=df[df['chargeOn']==False]['speed_self_cal'].describe().values[3]
        wire_cut_time=df[df['chargeOn']==False]['time_diff_cons_pings'].sum()
        min_voltage_pct_when_wire_cut=df[df['chargeOn']==False]['voltPct'].describe().values[3]
        print("charge_off_Speed:"+str(charge_off_Speed)+" ,wire_cut_time:"+str(wire_cut_time)+" ,min_voltage_pct_when_wire_cut:"+str(min_voltage_pct_when_wire_cut))

        #### CHARGE CONNECTED TO IGN ####
    
        charge_to_ign=(len(list(df['chargeOn'].eq(df['engineOn']).value_counts().values))==1)&(list(df['chargeOn'].eq(df['engineOn']).value_counts().keys())[0]==True)
        #### LOOSE WIRE ####
        fluctuations=np.sum(abs(df['chargeOn']-df['chargeOn'].shift(1)))
    
        #### IGN WIRE CUT ####
        ign_wire_cut=(len(df[df['engineOn']==False][df['speed_self_cal']>10])>20) & (len(df[df['engineOn']==True])<1)
    
        #### IGN WIRE TO CHARGE ####
        ign_to_charge=(len(df[df['chargeOn']==True][df['engineOn']==False])==0 )
    
        intentional_wire_cut=(charge_off_Speed>10)&(wire_cut_time>3600)&(min_voltage_pct_when_wire_cut<60)
        maintainence_wire_cut=(charge_off_Speed<2)&(wire_cut_time>3600)&(min_voltage_pct_when_wire_cut<60)
        wire_loose=fluctuations>2
        vehicle_id=df.vehicleId.iloc[0]
        values=[day,vehicle_id,intentional_wire_cut,maintainence_wire_cut,wire_loose,charge_to_ign,ign_wire_cut,ign_to_charge]
        columns=['day','vehicle_id','intentional_wire_cut','maintainence_wire_cut','wire_loose','charge_to_ign','ign_wire_cut','ign_to_charge']
    
        wire_issues=pd.DataFrame([values],columns=columns)
        #print("______________ wireissue main gaya_________________")
        return wire_issues
 #   else:
        
    except Exception as e:
        print("The issue with the wire_issue_hb_lc is--- "+ str(e))
        return None
    


# In[11]:




def lc_only_wire_issue(dfc):
    df=dfc.copy()
   
  #  if 1>0:
    try:

        df.sort_values('time',ascending=False,inplace=True)

        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(1)
        df=df[1:]
        time_diff_cons=df['time1']-df['time']
        df.loc[:,'time_diff_cons_pings']=time_diff_cons
   
   
        loc_list=df[['location','loc2']].values.tolist()

        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)
 
        df.loc[:,'distance']=dis
   
   
   
        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)
   
        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()

        day=getDay((df['time'].min()+df['time'].max())/2)
   
   
        #### WIRE CUT KE PARAMETERS ####
        charge_off_Speed=df[df['chargeOn']==False]['speed_self_cal'].describe().values[3]
        wire_cut_time=df[df['chargeOn']==False]['time_diff_cons_pings'].sum()
        min_voltage_pct_when_wire_cut=df[df['chargeOn']==False]['voltPct'].describe().values[3]
        print(min_voltage_pct_when_wire_cut)

   
        #### LOOSE WIRE ####
        fluctuations=np.sum(abs(df['chargeOn']-df['chargeOn'].shift(1)))

        #### CHARGE CONNECTED TO IGN ####
        charge_to_ign=(len(list(df['chargeOn'].eq(df['ignOn']).value_counts().values))==1)&(list(df['chargeOn'].eq(df['ignOn']).value_counts().keys())[0]==True)&(fluctuations>2)

   
        #### IGN WIRE CUT ####
        ign_wire_cut=(len(df[df['ignOn']==False][df['speed_self_cal']>10])>20) & (len(df[df['ignOn']==True])<1)
   
        #### IGN WIRE TO CHARGE ####
        ign_to_charge=(len(df[df['chargeOn']==True][df['ignOn']==False])==0 )&(fluctuations<3)
   
        intentional_wire_cut=(charge_off_Speed>12/3.6)&(wire_cut_time>3600)&(min_voltage_pct_when_wire_cut<60)
        maintainence_wire_cut=(charge_off_Speed<2/3.6)&(wire_cut_time>3600)&(min_voltage_pct_when_wire_cut<60)
        wire_loose=fluctuations>2
        vehicle_id=df.vehicleId.iloc[0]
        values=[day,vehicle_id,intentional_wire_cut,maintainence_wire_cut,wire_loose,charge_to_ign,ign_wire_cut,ign_to_charge]
        columns=['day','vehicle_id','intentional_wire_cut','maintainence_wire_cut','wire_loose','charge_to_ign','ign_wire_cut','ign_to_charge']
   
        wire_issues=pd.DataFrame([values],columns=columns)
       
        return wire_issues
#    else:
           
    except Exception as e :
        print("The issue with the wire_issue_hb_lc is--- "+ str(e))

        return None


def no_data(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    day=getDay((day_start+day_end)/2)

    
    final_values=[vid,day,0,0]+[-1]*46
    final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                        'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']

    day_wise=pd.DataFrame([final_values],columns=final_column_names)
    return day_wise

In [ ]:
def find_last_ping(vid):
    try:
        url=GPS_API_BASE+"/gps/top?vehicleIds={}&sortOrder=desc".format(vid)
        data=json.load(urllib.request.urlopen(url))
        last_loc=data['data'][str(vid)]['time']
    except:
        last_loc=0
    try:
        url=GPS_API_BASE+"/heartbeat/top?vehicleIds={}&sortOrder=desc".format(vid)
        data=json.load(urllib.request.urlopen(url))
        last_hb=data['data'][str(vid)]['time']
    except:
        last_hb=0
        
        
    if last_hb>=last_loc:
        last_ping=last_hb
    else:
        last_ping=last_loc
    
    if  last_ping>0:
        l=[[vid,last_ping-86400,last_ping+100]]
    else:
        l=[[vid,0,200]]

    return l

In [ ]:
# In[14]:


def hb_only(hbc,l):
    hb = hbc.copy()
 
    try:
        
        vid=l[0]
        day_start=l[1]
        day_end=l[2]

        hb['created']=hb['created']/1000
        total_counts=0

        #print('*** hb_only main hb pings= '+str(len(hb)))
        no_info_distance_travelled=None


        if('usedSat' in hb):
            sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]

        else:
            sat_used_values=[-1]*5



        if('gsmPct' in hb):
            gsm_pct_values=hb['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]

        else:
            gsm_pct_values=[-1]*5


        time_diff_cons_pings_values=[-1]*5

        server_delay_values=[-1]*6


        jump_values=[-1]*6

        driving_count=-1

        stop_count=-1


        driving_time=-1
        stop_time=-1

        day=getDay((day_start+day_end)/2)

        total_distace_travelled=-1


        hb=hb.sort_values('created',ascending=False)
        hb.reset_index(drop=True,inplace=True)
        hb['created_time_shift']=hb['created'].shift(1)
        hb.at[0,'created_time_shift']=hb.iloc[0]['created']
        hb['created_cons_time_diff']=hb['created_time_shift']-hb['created']
        created_cons_time_diff_values=hb[hb['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]

        live_time=hb[hb['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
        offline_time=hb[hb['created_cons_time_diff']>180]['created_cons_time_diff'].sum()

        if (hb.created.min()-day_start)>=180:
            offline_time=offline_time+hb.created.min()-day_start
        else:
            live_time=live_time+hb.created.min()-day_start


        if (day_end-hb.created.max())>=180:
            offline_time=offline_time+day_end-hb.created.max()
        else :
            live_time=live_time+day_end-hb.created.max()

        no_info_duration=hb[hb['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()

        if (hb.created.min()-day_start)>=3600:
            no_info_duration=no_info_duration+hb.created.min()-day_start

        elif (day_end-hb.created.max())>=3600:
            no_info_duration=no_info_duration+day_end-hb.created.max()


        jump_distance=-1
        jump_percent=-1

        expected_run_pings_percent=-1
        expected_stop_pings_percent=-1

        heart_beat_pings=len(hb)

        device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))



        no_info_instances=len(device_history)

        final_values=[vid,day,total_counts,heart_beat_pings]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration,no_info_instances]
        final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                        'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']


        day_wise=pd.DataFrame([final_values],columns=final_column_names)
        #print(day_wise.values())

        return day_wise
    except Exception as e:
        print(" ERROR IN HB ONLY ---------- "+str(e)+" for {}".format(l))
        
        return pd.DataFrame()


# In[15]:



def lc_only(dfc,l):

    df = dfc.copy()
#if (1>0):
    try:
        
        vid=l[0]
        day_start=l[1]
        day_end=l[2]

#        print('*** lc_only main lc pings= '+str(len(df)))
        df.sort_values('time',ascending=False,inplace=True)

        df.reset_index(drop=True,inplace=True)

        try:
            no_info_epochs=[]

            device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))

            if len(device_history)>0:
                device_history.sort_values('fromTime',ascending=True,inplace=True)
                no_info_epochs=[]
                for b,brow in device_history.iterrows():
                    try:
                        
                        x=df.sort_values('time')
                        top_x=x[df['time']<brow['fromTime']].head(1)
                        no_info_start=top_x['time'].iloc[0]
                        
                        
                    except:
                        no_info_start=999999999999
                    try:
                        y=df.sort_values('time')
                        top_y=y[df['time']>brow['toTime']].tail(1)
                        no_info_end=top_y['time'].iloc[0]

                    except:
                        no_info_end=0

                    no_info_epochs.append([no_info_start,no_info_end])

            else:
                no_info_distance_travelled=None

        except Exception as e:
            device_history=pd.DataFrame()

            no_info_distance_travelled=None
            print('for the no_info_error for lc_only is ' + str(e))



        total_counts=len(df)



        if('usedSat' in df):
            sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        else:
            sat_used_values=[-1]*5



        if('gsmPct' in df):
            gsm_pct_values=df['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        else:
            gsm_pct_values=[-1]*5


        df.sort_values('time',ascending=False,inplace=True)

        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(1)

        df=df[1:]

        time_diff_cons=df['time1']-df['time']

        df.loc[:,'time_diff_cons_pings']=time_diff_cons

        time_diff_cons_pings_values=df.time_diff_cons_pings.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]



        df.loc[:,'server_created_time']=df['created']/1000 #.apply(getTime)



        df.loc[:,'server_delay']=df['server_created_time']-df['time']

        server_delay_values=df.server_delay.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]


        loc_list=df[['location','loc2']].values.tolist()

        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)

        df.loc[:,'distance']=dis

        no_info_distance_travelled=0
        for epoch in no_info_epochs:
            no_info_distance_travelled+=df[df['time']>=epoch[0]][df['time']<=epoch[1]]['distance'].sum()

        jump_values=df[df['distance']>500]['distance'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]


        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)

        driving_count=len(df[df['moving_status']=='running'])

        stop_count=len(df[df['moving_status']=='stop'])


        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()

        day=getDay((df['time'].min()+df['time'].max())/2)

        total_distace_travelled=df['distance'].sum()


        df=df.sort_values('server_created_time',ascending=False)

        df.reset_index(drop=True,inplace=True)


        df['created_time_shift']=df['server_created_time'].shift(1)
        df.at[0,'created_time_shift']=df.iloc[0]['server_created_time']


        df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']


        live_time=df[df['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
        offline_time=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].sum()

        created_cons_time_diff_values=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]



        if (df.time.min()-day_start)>=180:
            offline_time=offline_time+df.time.min()-day_start
        else:
            live_time=live_time+df.time.min()-day_start


        if (day_end-df.time.max())>=180:
            offline_time=offline_time+day_end-df.time.max()
        else :
            live_time=live_time+day_end-df.time.max()


        jump_distance=df[df['distance']>500]['distance'].sum()
        jump_percent=100*jump_distance/total_distace_travelled

        expected_run_pings_percent=1000*driving_count/driving_time
        expected_stop_pings_percent=3000*stop_count/stop_time


        no_info_duration=df[df['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()

        if (df.time.min()-day_start)>=3600:
            no_info_duration=no_info_duration+df.time.min()-day_start

        if (day_end-df.time.max())>=3600:
            no_info_duration=no_info_duration+day_end-df.time.max()

        no_info_instances=len(device_history)




        final_values=[vid,day,total_counts,None]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration,no_info_instances]


        final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                 'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']


        day_wise=pd.DataFrame([final_values],columns=final_column_names)


        return day_wise
    #else:
    except Exception as e:
        print("LC ONLY MAIN ISSUE AAYA HAI------------" +str(e)+" for {}".format(l) )
        day_wise=pd.DataFrame()
        return day_wise



# In[16]:


def hb_lc(hbc,dfc,l):
    hb = hbc.copy()
    df = dfc.copy()
    try:

        vid=l[0]
        day_start=l[1]
        day_end=l[2]    

        hb['created']=hb['created']/1000
        total_counts=len(df)

#        print('***hb_lc main loc pings= '+str(len(df)))
        df=df.sort_values('time',ascending=False)
        df.reset_index(drop=True,inplace=True)

        try:
            no_info_epochs=[]
            device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))

            if len(device_history)>0:
                device_history.sort_values('fromTime',ascending=True,inplace=True)
                for b,brow in device_history.iterrows():
                    try:
                        x=df.sort_values('time')
                        top_x=x[df['time']<brow['fromTime']].head(1)
                        no_info_start=top_x['time'].iloc[0]


#                       no_info_start=df.sort_values('time')[df['time']<brow['fromTime']].head(1)['time'].iloc[0]
                    except:
                        no_info_start=999999999999
                    try:
                        y=df.sort_values('time')
                        top_y=y[df['time']>brow['toTime']].tail(1)
                        no_info_end=top_y['time'].iloc[0]

#                       no_info_end=df.sort_values('time')[df['time']>brow['toTime']].tail(1)['time'].iloc[0]

                    except:
                        no_info_end=0

                    no_info_epochs.append([no_info_start,no_info_end])

            else:
                no_info_distance_travelled=None

        except Exception as e:
            device_history=pd.DataFrame()
            no_info_distance_travelled=None
            print('for the no_info_error for hb_lc is ' + str(e))



        if('usedSat' in df):
            sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        elif('usedSat' in hb):
            sat_used_values=hb['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        else:
            sat_used_values=[-1]*5



        if('gsmPct' in hb):
            gsm_pct_values=hb['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        elif ('gsmPct' in df):
            gsm_pct_values=df['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        else:
            gsm_pct_values=[-1]*5


        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(1)

        df=df[1:]

        time_diff_cons=df['time1']-df['time']

        df.loc[:,'time_diff_cons_pings']=time_diff_cons

        time_diff_cons_pings_values=df.time_diff_cons_pings.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]


        df.loc[:,'server_created_time']=df['created']/1000 #.apply(getTime)



        df.loc[:,'server_delay']=df['server_created_time']-df['time']

        server_delay_values=df.server_delay.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]


        loc_list=df[['location','loc2']].values.tolist()

        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)

        df['distance']=dis

        no_info_distance_travelled=0
        for epoch in no_info_epochs:
            no_info_distance_travelled+=df[df['time']>=epoch[0]][df['time']<=epoch[1]]['distance'].sum()


        jump_values=df[df['distance']>500]['distance'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]


        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)

        driving_count=len(df[df['moving_status']=='running'])

        stop_count=len(df[df['moving_status']=='stop'])


        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()

        day=getDay((df['time'].min()+df['time'].max())/2)

        total_distace_travelled=df['distance'].sum()


        df=df.sort_values('server_created_time',ascending=False)

        df.reset_index(drop=True,inplace=True)
        df['created_time_shift']=df['server_created_time'].shift(1)
        df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']

        hb=hb.sort_values('created',ascending=False)
        hb.reset_index(drop=True,inplace=True)
        hb['created_time_shift']=hb['created'].shift(1)
        hb.at[0,'created_time_shift']=hb.iloc[0]['created']
        hb['created_cons_time_diff']=hb['created_time_shift']-hb['created']

        live_time=hb[hb['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
        offline_time=hb[hb['created_cons_time_diff']>180]['created_cons_time_diff'].sum()

        if (hb.created.min()-day_start)>=180:
            offline_time=offline_time+hb.created.min()-day_start
        else:
            live_time=live_time+hb.created.min()-day_start


        if (day_end-hb.created.max())>=180:
            offline_time=offline_time+day_end-hb.created.max()
        else :
            live_time=live_time+day_end-hb.created.max()

        no_info_duration=hb[hb['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()

        if (df.time.min()-day_start)>=3600:
            no_info_duration=no_info_duration+df.time.min()-day_start

        elif (day_end-df.time.max())>=3600:
            no_info_duration=no_info_duration+day_end-df.time.max()


        df['created_time_shift']=df['server_created_time'].shift(1)
        df.at[0,'created_time_shift']=df.iloc[0]['server_created_time']
        df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']
        created_cons_time_diff_values=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]
        jump_distance=df[df['distance']>500]['distance'].sum()
        jump_percent=100*jump_distance/total_distace_travelled

        expected_run_pings_percent=1000*driving_count/driving_time
        expected_stop_pings_percent=3000*stop_count/stop_time

        heart_beat_pings=len(hb)
        no_info_instances=len(device_history)
        final_values=[vid,day,total_counts,heart_beat_pings]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration,no_info_instances]
        final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']


        day_wise=pd.DataFrame([final_values],columns=final_column_names)
        
    

        return day_wise
    except Exception as e:
        print("HB_LC MAIN ISSUE AAYA HAI---------" + str(e)+" for  {}".format(l))
        day_wise=pd.DataFrame()
        return day_wise


def lambda_handler(event,context=None):
    if 1>0:
    
#    try:
        k=0
        vehicle_list=event
    
                
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(10)
        reuslts=pool.map(find_last_ping,vehicle_list)
        pool.close()
        pool.join()

        vid_start_end_draft=[]
        for i in reuslts:
            for p in i:
                vid_start_end_draft.append(p)
        
        vid_start_end=[]
        for i in vid_start_end_draft:
            if (i not in vid_start_end)&(isinstance(i,list)==True):
                vid_start_end.append(i)
        
        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S') +' :total iterations to go - '+str(len(vid_start_end)))
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(100)
        results2=pool.map(analysis_flow,vid_start_end)
        pool.close()
        pool.join()
        
        data=pd.DataFrame()
        wire_data=pd.DataFrame()
        for x in results2:
            if x!=None:
                
                wire_data=pd.concat([wire_data,x[1]])
                data=pd.concat([data,x[0]])
    

        db = client.get_database('telematics_db')
        
                                                                                                                     

                                                                                                                      
        data=data.round(0)

        vehicle_details=pd.read_sql("select vehicle_id,boarding_time,vehicle_number,data_source,operator_id,new_device,installation_subtype,device_id,is_force_live,operator_name,operator_phone_number,sim_id,model_name,sales_dept,sim_number,sim_phone_number,imei from vehicle_details",con=engine)

        data=pd.merge(data,vehicle_details,on='vehicle_id',how='left')
        wire_data=pd.merge(wire_data,vehicle_details,on='vehicle_id',how='left')

        devices=pd.read_sql("select created as device_created_time,imei,new_device as device_table_new_device,version as device_version,purchase_date as device_purchase_date from devices",con=mysql_vds)

        data=pd.merge(data,devices, on='imei',how='left')
        wire_data=pd.merge(wire_data,devices, on='imei',how='left')


        print((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S')+"  " +str(k)+ "range vehicles have total data for:"+str(len(data))+" vehicles to be appended")

#        wire_data.set_index(['vehicle_number','day','vehicle_id']).to_sql("wire_issues_no_info_snapshot",con=engine,if_exists='append',chunksize=200)

#        data.set_index(['vehicle_number','day','vehicle_id']).to_sql("device_health_no_info_snapshot",con=engine,if_exists='append',chunksize=200)
#        print(data)
        print("{}  ***************** 1000 batch done **************** ".format((dt.datetime.now()+timedelta(hours=5.5)).strftime('%m/%d %H:%M:%S')))

#    except Exception as e:
    else:
        print("KOI ISSUE AAYA HAI, VEHICLES NAHI HUE HONGE, the error being >>"+ str(e))
        data=pd.DataFrame()
    return (data,wire_data)

In [ ]:
lambda_handler([12,34,55])

In [ ]:
complete_wire=pd.DataFrame()
device_health=pd.DataFrame()
vehicles_clusters=[vehicles[i:i+100] for i in range (0,len(vehicles),100)]




In [ ]:
for j in vehicles_clusters:
    from multiprocessing.dummy import Pool as ThreadPool
    pool = ThreadPool(5)
    results2=pool.map(lambda_handler,[j])
    pool.close()
    pool.join()
    for i in results2:
        if i!=None:
            omplete_wire=pd.concat([complete_wire,i[1]])
            device_health=pd.concat([device_health,i[0]])



In [ ]:
com=complete_wire.drop_duplicates(['vehicle_id'])

complete_wire.to_csv("100_snapshot_wire_issues_run3.csv")

In [ ]:
omplete_wire.shape

In [ ]:
com[(com['intentional_wire_cut']==True) | (com['maintainence_wire_cut']==True)|(com['wire_loose']==True)|(com['charge_to_ign']==True)|(com['ign_wire_cut']==True)|(com['ign_to_charge']==True)]     





# 100 SAMPLE DATA

In [ ]:

def find_last_ping_data(vid):
    #if 1>0:
    try:
        try:
            url_gps=GPS_API_BASE+"/gps/top?vehicleIds={}&sortOrder=desc".format(vid)
            data_gps=json.load(urllib.request.urlopen(url_gps))
            print("this is the raw gps   "+ str(data_gps))
        except Exception as e:
            print("gps_issue : "+str(e))
            data_gps={}
        print(" _______________________ ")
        try:
            url_hb=GPS_API_BASE+"/heartbeat/top?vehicleIds={}&sortOrder=desc".format(vid)
            data_hb=json.load(urllib.request.urlopen(url_hb))
            print("this is the heart beat   "+ str(data_hb))
        except Exception as e :
            print("hb_issue : "+str(e))
            data_hb={}

        if (len(data_gps)>0)&(len(data_hb['data'])>0):
            if data_hb['data'][str(vid)]['created']/1000>data_gps['data'][str(vid)]['time']:
                last_ping_time=data_hb['data'][str(vid)]['created']/1000
                
                
                
                
                
            else:
                last_ping_time=data_gps['data'][str(vid)]['time']
                
            
            last_ping_location=data_gps['data'][str(vid)]['location']
            try:
                usedSat=data_gps['data'][str(vid)]['usedSat']
            except:
                usedSat=None
            speed=data_gps['data'][str(vid)]['speed']
            charge_state=data_hb['data'][str(vid)]['chargeOn']
            
            ign=data_hb['data'][str(vid)]['engineOn']
            voltage=data_hb['data'][str(vid)]['voltPct']
            gsm=data_hb['data'][str(vid)]['gsmPct']

        elif (len(data_gps)>0)&(len(data_hb['data'])<1):
            last_ping_time=data_gps['data'][str(vid)]['time']
            last_ping_location=data_gps['data'][str(vid)]['location']
            
            try:
                usedSat=data_gps['data'][str(vid)]['usedSat']
            except:
                usedSat=None
                
            try:
                speed=data_gps['data'][str(vid)]['speed']
            except:
                speed=None

            try:
                charge_state=data_gps['data'][str(vid)]['chargeOn']
            except:
                charge_state=None
                
            try:
                ign=data_gps['data'][str(vid)]['ignOn']
                
            except:
                ign=None
            try:
                voltage=data_gps['data'][str(vid)]['voltPct']
            except:
                voltage=None
            try:
                gsm=data_gps['data'][str(vid)]['gsmPct']
            except:
                gsm=None
        
                

        elif (len(data_gps)<1)&(len(data_hb)<1):
            last_ping_time=None
            last_ping_location=None
            charge_state=None
            ign=None
            voltage=None
            gsm=None
        
        try:
            last_state=pd.DataFrame(list(db2.itinerary.find({'vehicleId':vid}).sort('fromTime',-1).limit(2)))
            last_state_duration=last_state['totalTime'].iloc[1]
            last_state_mode=last_state['mode'].iloc[1]
        except:
            last_state_duration=None
            last_state_mode=None
            

        l=[vid,last_ping_time,last_ping_location,charge_state,ign,voltage,gsm,speed,usedSat,last_state_duration,last_state_mode]

        print(l)
  #  else:
        
    except Exception as e:
        print("the issue is "+ str(e))
        l=[None]*9
    return l


In [ ]:
vid=100001           
last_state=pd.DataFrame(list(db2.itinerary.find({'vehicleId':vid}).sort('fromTime',-1)))
last_state_duration=last_state['totalTime'].iloc[1]
last_state_mode=last_state['mode'].iloc[1]


In [ ]:
last_state

In [ ]:
final=[]
count=0
for i in vehicles:
    count+=1
    l=find_last_ping_data(i)
    final.append(l)
    print(count)
last_db=pd.DataFrame(final,columns=['vehicle_id','last_ping_time','last_ping_location','charge_state','ign','voltage','gsm','speed','usedSat','last_state_duration','last_state_mode'])    
    

In [ ]:
devices=pd.read_sql("select d.created,vd.vehicle_id from devices d,vehicle_devices vd where vd.vehicle_id in {} and d.id=vd.device_id and vd.is_latest=1".format(tuple(vehicles)),con=mysql_vds)   
vehicle_details=pd.read_sql("select vehicle_number,vehicle_id,model_name,live_time,sub_type_installation,network,sales_dept,operator_phone_number,operator_name from vehicle_details where vehicle_id in {} ".format(tuple(vehicles)),con=engine)      
last_db=pd.merge(last_db,vehicle_details,on='vehicle_id',how='left')
last_db=pd.merge(last_db,devices,on='vehicle_id',how='left')
last_db.to_csv("last_db_data_run3.csv")
#data['diffrence']=tm.time()-data['last_ping_time']



In [ ]:
no_info=pd.DataFrame(list(db.deviceHistory.find({'toTime':None})))

In [ ]:
no_info['toTime']=[tm.time()]*len(no_info)

In [ ]:
no_info['diffrence']=no_info['toTime']-no_info['fromTime']

In [ ]:
vehicle_details=pd.read_sql("select vehicle_number,vehicle_id,model_name,live_time,sub_type_installation,network,sales_dept,operator_phone_number,operator_name from vehicle_details where vehicle_id in {} ".format(tuple(no_info['vehicleId'])),con=engine)      


In [ ]:
no_info=pd.merge(no_info,vehicle_details,left_on='vehicleId',right_on='vehicle_id',how='left')

In [ ]:
no_info=no_info[no_info['model_name'].isin(['TL250','WETRACK2','FMB920','C100'])]

In [ ]:
selected_vehicles_df=pd.read_csv("selected_vehicles_for_snapshot_run2.csv")
selected_vehicles_df['vehicle_id']=selected_vehicles_df['vehicle_id'].apply(int)
old_vehicles=selected_vehicles_df['vehicle_id']


In [ ]:
no_info=no_info[~no_info['vehicle_id'].isin(old_vehicles)]

In [ ]:
bucket=[]
for i in list(no_info['diffrence']):
    if (i<4*3600):
        bucket.append("less than 4 hours")
    elif (i>=4*3600)&(i<8*3600):
        bucket.append(" 4-8 hours")
    elif (i>=8*3600)&(i<12*3600):
        bucket.append(" 8-12 hours")
    elif (i>=12*3600)&(i<16*3600):
        bucket.append("12-16 hours")
    elif (i>=16*3600)&(i<20*3600):
        bucket.append("16-20 hours")
    elif (i>=20*3600)&(i<24*3600):
        bucket.append("20-24 hours")    
    elif (i>=1*86400)&(i<3*86400):
        bucket.append("1-3 days")
    elif(i>=3*86400)&(i<7*86400):
        bucket.append("3-7 days")
    elif(i>=7*86400)&(i<15*86400):
        bucket.append("7-15 days")
    elif(i>=15*86400)&(i<30*86400):
        bucket.append("15-30 days")
    elif (i>=30*86400):
        bucket.append("30+ days")
no_info['bucket']=bucket   

In [ ]:
no_info.shape
no_info.drop_duplicates('vehicleId').shape

In [ ]:
no_info.bucket.value_counts()

In [ ]:
size = 300        # sample size
replace = False  # with replacement
fn = lambda obj: obj.loc[np.random.choice(obj.index, size, replace),:]
selected_vehicles_df=no_info[~no_info['bucket'].isin( [' 4-8 hours','less than 4 hours','12-16 hours','20-24 hours','16-20 hours'])].groupby('bucket', as_index=False).apply(fn)


In [ ]:
selected_vehicles_df=pd.read_csv("selected_vehicles_for_snapshot_run3.csv")
selected_vehicles_df['vehicle_id']=selected_vehicles_df['vehicle_id'].apply(int)

In [ ]:
selected_vehicles_df.to_csv("selected_vehicles_for_snapshot_run3.csv")

In [ ]:
vehicles=list(selected_vehicles_df['vehicle_id'])

In [ ]:
len(vehicles)-len(set(vehicles))

In [ ]:
selected_vehicles_df['bucket'].value_counts()

# API HITS

In [ ]:
for vid in vehicles:
    url="https://example.ngrok.io/device/populateNoInfoDevice?vehicleID={}".format(vid)
    json.load(urllib.request.urlopen(url))
